# CS5224 Data Curation — Stage 1E (Revised)

## Data-quality flags, catalog eligibility, and offer availability

This revised Stage 1E separates **catalog eligibility** from **current stock availability**.

A product/listing can remain searchable even when it is out of stock.

### Key distinction

- `catalog_eligible`:
  the record is sufficiently clean to remain in the searchable catalog.
- `currently_available`:
  whether the source explicitly says the product is currently in stock.
- `offer_eligible`:
  the record is catalog-eligible **and** currently in stock, so it can be used as an active purchasable offer.

Therefore a clean out-of-stock record can have:

```text
catalog_eligible      = True
stock_status          = OUT_OF_STOCK
currently_available   = False
offer_eligible        = False
```

This lets CartOptix return the item in search while correctly showing that it is unavailable for purchase.

### Stage 1E quality dimensions

- brand quality;
- price quality;
- stock status;
- specification quality.

### Stage 1E adds

- `brand_status`
- `price_status`
- `stock_status`
- `spec_quality_status`
- `quality_issue_codes`
- `hard_exclusion_reason`
- `needs_manual_review`
- `basic_eligibility`
- `catalog_eligible`
- `currently_available`
- `offer_eligible`
- `eligibility_status`

`ELIGIBLE_PRELIMINARY` is still **not final seed approval**. Later stages must still check category, direct supermarket fulfilment, exact matching, promotions, delivery rules, and final verification.


## 1. Imports and project paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"

STAGE1D_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1d_spec_parsed.csv"
)

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nStaging data directory:")
print(STAGING_DATA_DIR)

print("\nStage 1D input:")
print(STAGE1D_FILE)

print("\nExists:", STAGE1D_FILE.exists())

assert STAGE1D_FILE.exists(), (
    "Stage 1D checkpoint not found. "
    "Run Data_Curation_Stage1D.ipynb first."
)


Current notebook folder:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\scripts

Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Staging data directory:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging

Stage 1D input:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1d_spec_parsed.csv

Exists: True


## 2. Define expected Stage 1D schema and counts

In [2]:
STAGE1D_COLUMNS = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",
    "source_category_raw",
    "source_category_normalized",
    "product_name_raw",
    "product_name_normalized",
    "brand_raw",
    "brand_normalized",
    "specification_raw",
    "specification_normalized",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
    "spec_parse_notes",
    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",
    "savings_raw",
    "stock_raw",
    "in_stock",
    "offer_raw",
    "has_offer",
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
]

EXPECTED_TOTAL_ROWS = 177_638

EXPECTED_RETAILER_COUNTS = {
    "redmart": 97_779,
    "fairprice": 69_402,
    "coldstorage": 10_457,
}

VALID_SPEC_STATUSES = {
    "PARSED",
    "PARTIAL",
    "AMBIGUOUS",
    "MISSING",
}


## 3. Load the Stage 1D checkpoint

In [3]:
stage1d = pd.read_csv(
    STAGE1D_FILE,
    dtype={
        "retailer": "string",
        "source_file": "string",
        "source_sheet": "string",
        "source_record_id": "string",
        "source_category_raw": "string",
        "source_category_normalized": "string",
        "product_name_raw": "string",
        "product_name_normalized": "string",
        "brand_raw": "string",
        "brand_normalized": "string",
        "specification_raw": "string",
        "specification_normalized": "string",
        "unit": "string",
        "base_unit": "string",
        "spec_parse_status": "string",
        "spec_parse_pattern": "string",
        "spec_parse_notes": "string",
        "stock_raw": "string",
        "offer_raw": "string",
        "image_url_raw": "string",
        "product_url_raw": "string",
        "comments_raw": "string",
    }
)

stage1d["source_row_number"] = (
    pd.to_numeric(
        stage1d["source_row_number"],
        errors="coerce"
    )
    .astype("Int64")
)

stage1d["pack_count"] = (
    pd.to_numeric(
        stage1d["pack_count"],
        errors="coerce"
    )
    .astype("Int64")
)

for column in [
    "unit_size",
    "total_quantity",
    "total_quantity_base",
    "product_price",
    "original_price",
]:
    stage1d[column] = (
        pd.to_numeric(
            stage1d[column],
            errors="coerce"
        )
        .astype("Float64")
    )

def to_nullable_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "1": True,
        "0": False,
    }

    return (
        normalized
        .map(mapping)
        .astype("boolean")
    )

stage1d["in_stock"] = to_nullable_boolean(
    stage1d["in_stock"]
)

stage1d["has_offer"] = to_nullable_boolean(
    stage1d["has_offer"]
)

print("Loaded Stage 1D shape:")
print(stage1d.shape)

display(stage1d.head(3))


Loaded Stage 1D shape:
(177638, 34)


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,...,original_price_raw,original_price,savings_raw,stock_raw,in_stock,offer_raw,has_offer,image_url_raw,product_url_raw,comments_raw
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,...,NaN,<NA>,NaN,Yes,True,No,False,https://sg-live-01.slatic.net/p/5eaa6087b6c250...,https://www.lazada.sg/products/pdp-i303336401-...,<NA>
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,...,NaN,<NA>,Spend $45 + Free Gift,Yes,True,Yes,True,https://sg-test-11.slatic.net/p/9f86cfbce0b472...,https://www.lazada.sg/products/pdp-i462726209-...,<NA>
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,...,NaN,<NA>,NaN,Yes,True,No,False,https://sg-live-01.slatic.net/p/7579bbca2beaa0...,https://www.lazada.sg/products/pdp-i303250731-...,<NA>


## 4. Validate Stage 1D input

In [4]:
assert stage1d.columns.tolist() == STAGE1D_COLUMNS, (
    "Stage 1D columns do not match the expected schema."
)

assert len(stage1d) == EXPECTED_TOTAL_ROWS

assert (
    stage1d["source_record_id"]
    .isna()
    .sum()
    == 0
)

assert (
    stage1d["source_record_id"]
    .duplicated()
    .sum()
    == 0
)

actual_spec_statuses = set(
    stage1d[
        "spec_parse_status"
    ]
    .dropna()
    .unique()
)

assert (
    actual_spec_statuses
    <= VALID_SPEC_STATUSES
), (
    "Unexpected specification parse status found: "
    f"{actual_spec_statuses - VALID_SPEC_STATUSES}"
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1d["retailer"] == retailer
    ).sum()

    assert actual_count == expected_count, (
        f"{retailer}: expected {expected_count:,}, "
        f"found {actual_count:,}."
    )

print("PASS: Stage 1D input validation succeeded.")


PASS: Stage 1D input validation succeeded.


## 5. Preserve the complete Stage 1D dataset

In [5]:
stage1d_snapshot = stage1d.copy(deep=True)

print(
    f"Stage 1D snapshot created: "
    f"{stage1d_snapshot.shape[0]:,} rows x "
    f"{stage1d_snapshot.shape[1]} columns"
)


Stage 1D snapshot created: 177,638 rows x 34 columns


## 6. Build brand-quality status

Rules:

- `VALID`: normalized brand exists;
- `PLACEHOLDER`: raw source brand is `*`;
- `MISSING`: normalized brand is unavailable.

No missing brand is guessed automatically.


In [6]:
stage1e = stage1d.copy()

brand_raw_clean = (
    stage1e["brand_raw"]
    .astype("string")
    .str.strip()
)

brand_normalized_clean = (
    stage1e["brand_normalized"]
    .astype("string")
    .str.strip()
)

brand_placeholder_mask = (
    brand_raw_clean
    .eq("*")
    .fillna(False)
)

brand_missing_mask = (
    brand_normalized_clean.isna()
    | brand_normalized_clean.eq("").fillna(False)
)

stage1e["brand_status"] = pd.Series(
    np.select(
        [
            brand_placeholder_mask,
            brand_missing_mask,
        ],
        [
            "PLACEHOLDER",
            "MISSING",
        ],
        default="VALID",
    ),
    index=stage1e.index,
    dtype="string",
)

print("Brand status counts:")
print(
    stage1e["brand_status"]
    .value_counts(dropna=False)
)


Brand status counts:
brand_status
VALID          171214
MISSING          6200
PLACEHOLDER       224
Name: count, dtype: Int64


## 7. Build price-quality status

Rules:

- `VALID`: price > 0;
- `INVALID_ZERO`: price = 0;
- `INVALID_NEGATIVE`: price < 0;
- `INVALID_UNPARSED`: raw price exists but normalized price is missing;
- `MISSING`: no usable price was supplied.

Price problems are hard data-quality failures at Stage 1E.


In [7]:
price = stage1e["product_price"]

price_raw_present = (
    stage1e["product_price_raw"]
    .notna()
)

price_missing_mask = (
    price.isna()
    & ~price_raw_present
)

price_unparsed_mask = (
    price.isna()
    & price_raw_present
)

price_zero_mask = (
    price.eq(0)
    .fillna(False)
)

price_negative_mask = (
    price.lt(0)
    .fillna(False)
)

stage1e["price_status"] = pd.Series(
    np.select(
        [
            price_negative_mask,
            price_zero_mask,
            price_unparsed_mask,
            price_missing_mask,
        ],
        [
            "INVALID_NEGATIVE",
            "INVALID_ZERO",
            "INVALID_UNPARSED",
            "MISSING",
        ],
        default="VALID",
    ),
    index=stage1e.index,
    dtype="string",
)

print("Price status counts:")
print(
    stage1e["price_status"]
    .value_counts(dropna=False)
)


Price status counts:
price_status
VALID           177636
INVALID_ZERO         2
Name: count, dtype: Int64


## 8. Build stock status

Stock is treated as **availability information**, not as catalog-quality failure.

Rules:

- `IN_STOCK`
- `OUT_OF_STOCK`
- `UNKNOWN`


In [8]:
stage1e["stock_status"] = pd.Series(
    np.select(
        [
            stage1e["in_stock"].eq(True).fillna(False),
            stage1e["in_stock"].eq(False).fillna(False),
        ],
        [
            "IN_STOCK",
            "OUT_OF_STOCK",
        ],
        default="UNKNOWN",
    ),
    index=stage1e.index,
    dtype="string",
)

print("Stock status counts:")
print(
    stage1e["stock_status"]
    .value_counts(dropna=False)
)


Stock status counts:
stock_status
IN_STOCK        146553
OUT_OF_STOCK     31085
Name: count, dtype: Int64


## 9. Copy specification quality into an explicit Stage 1E field

In [9]:
stage1e["spec_quality_status"] = (
    stage1e["spec_parse_status"]
    .astype("string")
)

print("Specification quality counts:")
print(
    stage1e["spec_quality_status"]
    .value_counts(dropna=False)
)


Specification quality counts:
spec_quality_status
PARSED       139917
PARTIAL       27654
MISSING        6915
AMBIGUOUS      3152
Name: count, dtype: Int64


## 10. Define quality issue masks

`OUT_OF_STOCK` and `STOCK_UNKNOWN` are still recorded as issue/status codes for reporting, but they are **not hard exclusions from the searchable catalog**.

Only invalid/missing prices are hard Stage 1E exclusions.


In [10]:
ISSUE_MASKS = {
    "BRAND_PLACEHOLDER": (
        stage1e["brand_status"] == "PLACEHOLDER"
    ),
    "BRAND_MISSING": (
        stage1e["brand_status"] == "MISSING"
    ),
    "PRICE_INVALID_ZERO": (
        stage1e["price_status"] == "INVALID_ZERO"
    ),
    "PRICE_INVALID_NEGATIVE": (
        stage1e["price_status"] == "INVALID_NEGATIVE"
    ),
    "PRICE_INVALID_UNPARSED": (
        stage1e["price_status"] == "INVALID_UNPARSED"
    ),
    "PRICE_MISSING": (
        stage1e["price_status"] == "MISSING"
    ),
    "OUT_OF_STOCK": (
        stage1e["stock_status"] == "OUT_OF_STOCK"
    ),
    "STOCK_UNKNOWN": (
        stage1e["stock_status"] == "UNKNOWN"
    ),
    "SPEC_PARTIAL": (
        stage1e["spec_quality_status"] == "PARTIAL"
    ),
    "SPEC_AMBIGUOUS": (
        stage1e["spec_quality_status"] == "AMBIGUOUS"
    ),
    "SPEC_MISSING": (
        stage1e["spec_quality_status"] == "MISSING"
    ),
}

# Revised: stock does not hard-exclude a searchable catalog record.
HARD_EXCLUSION_CODES = {
    "PRICE_INVALID_ZERO",
    "PRICE_INVALID_NEGATIVE",
    "PRICE_INVALID_UNPARSED",
    "PRICE_MISSING",
}

print("Quality issue counts:")
for code_name, mask in ISSUE_MASKS.items():
    print(
        f"{code_name:25s}: "
        f"{int(mask.sum()):,}"
    )


Quality issue counts:
BRAND_PLACEHOLDER        : 224
BRAND_MISSING            : 6,200
PRICE_INVALID_ZERO       : 2
PRICE_INVALID_NEGATIVE   : 0
PRICE_INVALID_UNPARSED   : 0
PRICE_MISSING            : 0
OUT_OF_STOCK             : 31,085
STOCK_UNKNOWN            : 0
SPEC_PARTIAL             : 27,654
SPEC_AMBIGUOUS           : 3,152
SPEC_MISSING             : 6,915


## 11. Create quality issue code strings

In [11]:
def build_reason_codes(index, issue_masks, selected_codes=None):
    if selected_codes is None:
        selected_codes = list(
            issue_masks.keys()
        )

    reasons = pd.Series(
        "",
        index=index,
        dtype="string",
    )

    for code_name in selected_codes:
        mask = (
            issue_masks[code_name]
            .fillna(False)
        )

        existing = reasons.loc[mask]

        reasons.loc[mask] = np.where(
            existing.eq(""),
            code_name,
            existing + ";" + code_name,
        )

    reasons = reasons.mask(
        reasons.eq(""),
        "NONE"
    )

    return reasons


stage1e["quality_issue_codes"] = (
    build_reason_codes(
        stage1e.index,
        ISSUE_MASKS,
    )
)

stage1e["hard_exclusion_reason"] = (
    build_reason_codes(
        stage1e.index,
        ISSUE_MASKS,
        selected_codes=[
            code_name
            for code_name in ISSUE_MASKS
            if code_name in HARD_EXCLUSION_CODES
        ],
    )
)

display(
    stage1e[
        [
            "source_record_id",
            "quality_issue_codes",
            "hard_exclusion_reason",
        ]
    ].head(10)
)


,source_record_id,quality_issue_codes,hard_exclusion_reason
0,redmart_00000002,NONE,NONE
1,redmart_00000003,NONE,NONE
2,redmart_00000004,NONE,NONE
3,redmart_00000005,NONE,NONE
4,redmart_00000006,NONE,NONE
5,redmart_00000007,NONE,NONE
6,redmart_00000008,NONE,NONE
7,redmart_00000009,NONE,NONE
8,redmart_00000010,NONE,NONE
9,redmart_00000011,NONE,NONE


## 12. Define manual-review status

Manual review is driven by **data quality**, not simply by stock availability.

A clean out-of-stock product does not require manual correction merely because it is out of stock.


In [12]:
stage1e["needs_manual_review"] = (
    (
        stage1e["brand_status"]
        != "VALID"
    )
    |
    (
        stage1e["price_status"]
        != "VALID"
    )
    |
    (
        stage1e["spec_quality_status"]
        != "PARSED"
    )
).astype("boolean")

print("Manual-review flag:")
print(
    stage1e["needs_manual_review"]
    .value_counts(dropna=False)
)


Manual-review flag:
needs_manual_review
False    134126
True      43512
Name: count, dtype: Int64


## 13. Define basic data-quality eligibility

Stock is intentionally **not** part of this gate.

A record is quality-eligible when:

1. brand is valid;
2. product price is valid;
3. specification is fully parsed.


In [13]:
stage1e["basic_eligibility"] = (
    (
        stage1e["brand_status"]
        == "VALID"
    )
    &
    (
        stage1e["price_status"]
        == "VALID"
    )
    &
    (
        stage1e["spec_quality_status"]
        == "PARSED"
    )
).astype("boolean")

print("Basic quality eligibility:")
print(
    stage1e["basic_eligibility"]
    .value_counts(dropna=False)
)


Basic quality eligibility:
basic_eligibility
True     134126
False     43512
Name: count, dtype: Int64


## 14. Define catalog eligibility

At Stage 1E, a record is `catalog_eligible` when it passes the basic quality gate.

This means an out-of-stock record can still be retained for search/display.


In [14]:
stage1e["catalog_eligible"] = (
    stage1e["basic_eligibility"]
    .astype("boolean")
)

print("Catalog eligibility:")
print(
    stage1e["catalog_eligible"]
    .value_counts(dropna=False)
)


Catalog eligibility:
catalog_eligible
True     134126
False     43512
Name: count, dtype: Int64


## 15. Define current availability

- `True`: source explicitly says in stock;
- `False`: source explicitly says out of stock;
- `<NA>`: stock status is unknown.


In [15]:
stage1e["currently_available"] = pd.Series(
    pd.NA,
    index=stage1e.index,
    dtype="boolean",
)

stage1e.loc[
    stage1e["stock_status"] == "IN_STOCK",
    "currently_available"
] = True

stage1e.loc[
    stage1e["stock_status"] == "OUT_OF_STOCK",
    "currently_available"
] = False

print("Current availability:")
print(
    stage1e["currently_available"]
    .value_counts(dropna=False)
)


Current availability:
currently_available
True     146553
False     31085
Name: count, dtype: Int64


## 16. Define active offer eligibility

`offer_eligible` requires:

- catalog eligibility; and
- explicit current availability.

Therefore an out-of-stock product can remain in search but cannot be counted as a currently purchasable offer.


In [16]:
stage1e["offer_eligible"] = (
    stage1e["catalog_eligible"]
    &
    stage1e["currently_available"]
        .fillna(False)
).astype("boolean")

print("Active offer eligibility:")
print(
    stage1e["offer_eligible"]
    .value_counts(dropna=False)
)


Active offer eligibility:
offer_eligible
True     108997
False     68641
Name: count, dtype: Int64


## 17. Assign preliminary catalog eligibility status

- `ELIGIBLE_PRELIMINARY`: passes the Stage 1E catalog-quality gate;
- `REVIEW_REQUIRED`: price is usable, but brand/specification still requires review;
- `INELIGIBLE_HARD_GATE`: invalid/missing price.

Stock does not control these three catalog-quality statuses.


In [17]:
hard_fail_mask = (
    stage1e["hard_exclusion_reason"]
    != "NONE"
)

catalog_eligible_mask = (
    stage1e["catalog_eligible"]
    .eq(True)
    .fillna(False)
)

stage1e["eligibility_status"] = pd.Series(
    np.select(
        [
            hard_fail_mask,
            catalog_eligible_mask,
        ],
        [
            "INELIGIBLE_HARD_GATE",
            "ELIGIBLE_PRELIMINARY",
        ],
        default="REVIEW_REQUIRED",
    ),
    index=stage1e.index,
    dtype="string",
)

print("Eligibility status counts:")
print(
    stage1e["eligibility_status"]
    .value_counts(dropna=False)
)


Eligibility status counts:
eligibility_status
ELIGIBLE_PRELIMINARY    134126
REVIEW_REQUIRED          43510
INELIGIBLE_HARD_GATE         2
Name: count, dtype: Int64


## 18. Put Stage 1E fields into a clear column order

In [18]:
STAGE1E_COLUMNS = [
    # Provenance
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",

    # Category
    "source_category_raw",
    "source_category_normalized",

    # Product name
    "product_name_raw",
    "product_name_normalized",

    # Brand
    "brand_raw",
    "brand_normalized",
    "brand_status",

    # Specification
    "specification_raw",
    "specification_normalized",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
    "spec_parse_notes",
    "spec_quality_status",

    # Price
    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",
    "price_status",

    # Promotion
    "savings_raw",

    # Stock / availability
    "stock_raw",
    "in_stock",
    "stock_status",
    "currently_available",

    # Offer source flag
    "offer_raw",
    "has_offer",

    # URLs/comments
    "image_url_raw",
    "product_url_raw",
    "comments_raw",

    # Stage 1E quality / eligibility
    "quality_issue_codes",
    "hard_exclusion_reason",
    "needs_manual_review",
    "basic_eligibility",
    "catalog_eligible",
    "offer_eligible",
    "eligibility_status",
]

stage1e = stage1e[
    STAGE1E_COLUMNS
]

print("Number of Stage 1E columns:")
print(len(stage1e.columns))

for i, column in enumerate(
    stage1e.columns,
    start=1
):
    print(f"{i:2d}. {column}")

assert len(stage1e.columns) == 46


Number of Stage 1E columns:
46
 1. retailer
 2. source_file
 3. source_sheet
 4. source_row_number
 5. source_record_id
 6. source_category_raw
 7. source_category_normalized
 8. product_name_raw
 9. product_name_normalized
10. brand_raw
11. brand_normalized
12. brand_status
13. specification_raw
14. specification_normalized
15. pack_count
16. unit_size
17. unit
18. total_quantity
19. total_quantity_base
20. base_unit
21. spec_parse_status
22. spec_parse_pattern
23. spec_parse_notes
24. spec_quality_status
25. product_price_raw
26. product_price
27. original_price_raw
28. original_price
29. price_status
30. savings_raw
31. stock_raw
32. in_stock
33. stock_status
34. currently_available
35. offer_raw
36. has_offer
37. image_url_raw
38. product_url_raw
39. comments_raw
40. quality_issue_codes
41. hard_exclusion_reason
42. needs_manual_review
43. basic_eligibility
44. catalog_eligible
45. offer_eligible
46. eligibility_status


## 19. Verify all Stage 1D fields remain unchanged

In [19]:
assert stage1e[
    STAGE1D_COLUMNS
].equals(stage1d_snapshot), (
    "At least one Stage 1D field changed during Stage 1E."
)

print(
    "PASS: All 34 Stage 1D fields remain unchanged."
)


PASS: All 34 Stage 1D fields remain unchanged.


## 20. Validate brand-status logic

In [20]:
placeholder_rows = (
    stage1e["brand_status"]
    == "PLACEHOLDER"
)

missing_brand_rows = (
    stage1e["brand_status"]
    == "MISSING"
)

valid_brand_rows = (
    stage1e["brand_status"]
    == "VALID"
)

assert (
    stage1e.loc[
        placeholder_rows,
        "brand_raw"
    ]
    .astype("string")
    .str.strip()
    .eq("*")
    .all()
)

assert (
    stage1e.loc[
        valid_brand_rows,
        "brand_normalized"
    ]
    .notna()
    .all()
)

assert (
    stage1e.loc[
        placeholder_rows | missing_brand_rows,
        "brand_normalized"
    ]
    .isna()
    .all()
)

print(
    "PASS: Brand status correctly distinguishes "
    "valid, missing and '*' placeholder brands."
)


PASS: Brand status correctly distinguishes valid, missing and '*' placeholder brands.


## 21. Validate price-status logic

In [21]:
valid_price_rows = (
    stage1e["price_status"]
    == "VALID"
)

zero_price_rows = (
    stage1e["price_status"]
    == "INVALID_ZERO"
)

negative_price_rows = (
    stage1e["price_status"]
    == "INVALID_NEGATIVE"
)

assert (
    stage1e.loc[
        valid_price_rows,
        "product_price"
    ] > 0
).all()

assert (
    stage1e.loc[
        zero_price_rows,
        "product_price"
    ] == 0
).all()

assert (
    stage1e.loc[
        negative_price_rows,
        "product_price"
    ] < 0
).all()

print(
    "PASS: Price-quality statuses are internally consistent."
)


PASS: Price-quality statuses are internally consistent.


## 22. Inspect zero-price records explicitly

In [22]:
zero_price_review = stage1e.loc[
    stage1e[
        "price_status"
    ] == "INVALID_ZERO",
    [
        "retailer",
        "source_record_id",
        "source_category_raw",
        "product_name_raw",
        "brand_raw",
        "product_price_raw",
        "product_price",
        "stock_status",
        "product_url_raw",
        "price_status",
        "catalog_eligible",
        "offer_eligible",
        "eligibility_status",
    ]
].copy()

print(
    "Zero-price records:",
    f"{len(zero_price_review):,}"
)

display(zero_price_review)


Zero-price records: 2


,retailer,source_record_id,source_category_raw,product_name_raw,brand_raw,product_price_raw,product_price,stock_status,product_url_raw,price_status,catalog_eligible,offer_eligible,eligibility_status
110226,fairprice,fairprice_00012449,Chocolate Bars & Blocks,Meiji Chocolate Block - Black,Meiji,0.0,0.0,OUT_OF_STOCK,https://www.fairprice.com.sg/product/meiji-cho...,INVALID_ZERO,False,False,INELIGIBLE_HARD_GATE
164532,fairprice,fairprice_00066755,Broths & Stocks,Daisho Crab Miso Hot Pot Soup,Daisho,0.0,0.0,OUT_OF_STOCK,https://www.fairprice.com.sg/product/daisho-cr...,INVALID_ZERO,False,False,INELIGIBLE_HARD_GATE


## 23. Validate stock/availability logic

In [23]:
valid_stock_statuses = {
    "IN_STOCK",
    "OUT_OF_STOCK",
    "UNKNOWN",
}

actual_stock_statuses = set(
    stage1e[
        "stock_status"
    ]
    .dropna()
    .unique()
)

assert (
    actual_stock_statuses
    <= valid_stock_statuses
)

assert (
    stage1e.loc[
        stage1e["stock_status"] == "IN_STOCK",
        "currently_available"
    ]
    .eq(True)
    .all()
)

assert (
    stage1e.loc[
        stage1e["stock_status"] == "OUT_OF_STOCK",
        "currently_available"
    ]
    .eq(False)
    .all()
)

assert (
    stage1e.loc[
        stage1e["stock_status"] == "UNKNOWN",
        "currently_available"
    ]
    .isna()
    .all()
)

print("Stock status counts:")
print(
    stage1e["stock_status"]
    .value_counts(dropna=False)
)

print(
    "\nPASS: Stock status is retained separately "
    "from catalog eligibility."
)


Stock status counts:
stock_status
IN_STOCK        146553
OUT_OF_STOCK     31085
Name: count, dtype: Int64

PASS: Stock status is retained separately from catalog eligibility.


## 24. Validate that clean out-of-stock records remain catalog eligible

This is the key revised requirement.


In [24]:
clean_out_of_stock = (
    stage1e["catalog_eligible"].eq(True).fillna(False)
    &
    (stage1e["stock_status"] == "OUT_OF_STOCK")
)

clean_out_of_stock_count = int(
    clean_out_of_stock.sum()
)

print(
    "Catalog-eligible but out-of-stock records:",
    f"{clean_out_of_stock_count:,}"
)

if clean_out_of_stock_count > 0:
    assert (
        stage1e.loc[
            clean_out_of_stock,
            "offer_eligible"
        ]
        .eq(False)
        .all()
    )

    assert (
        stage1e.loc[
            clean_out_of_stock,
            "eligibility_status"
        ]
        == "ELIGIBLE_PRELIMINARY"
    ).all()

    display(
        stage1e.loc[
            clean_out_of_stock,
            [
                "retailer",
                "source_record_id",
                "product_name_raw",
                "brand_normalized",
                "product_price",
                "stock_status",
                "catalog_eligible",
                "currently_available",
                "offer_eligible",
                "eligibility_status",
            ]
        ].head(20)
    )

print(
    "\nPASS: Out-of-stock does not remove an otherwise "
    "clean record from the searchable catalog."
)


Catalog-eligible but out-of-stock records: 25,129


,retailer,source_record_id,product_name_raw,brand_normalized,product_price,stock_status,catalog_eligible,currently_available,offer_eligible,eligibility_status
75,redmart,redmart_00000077,Marigold Pudding - Custard Caramel 95G,marigold,1.68,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
76,redmart,redmart_00000078,Monoprix Gourmet Dark Chocolate Lava Cake,monoprix,8.5,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
77,redmart,redmart_00000079,DANONE Danette Chocolate,danone,9.89,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
78,redmart,redmart_00000080,Monoprix Gourmet Dark Chocolate Mousse 2X90G,monoprix,10.7,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
79,redmart,redmart_00000081,Monoprix Tiramisu With Mascarpone 2X80G,monoprix,7.6,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
80,redmart,redmart_00000082,TARAMI Pure Series Peach Jelly - Japan Special,tarami,3.97,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
81,redmart,redmart_00000083,La Laitiere Secret Chocolate Mousse 4 X 59G,la laitiere,6.9,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
82,redmart,redmart_00000084,DANONE Danette Vanilla 4x125g,danone,8.9,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
83,redmart,redmart_00000085,Arla Protein Vanilla Cookie Pudding 200G - Lac...,arla,4.3,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY
84,redmart,redmart_00000086,Marigold Jelly - Mango 125G,marigold,1.68,OUT_OF_STOCK,True,False,False,ELIGIBLE_PRELIMINARY



PASS: Out-of-stock does not remove an otherwise clean record from the searchable catalog.


## 25. Validate active offer eligibility

In [25]:
offer_eligible_mask = (
    stage1e["offer_eligible"]
    .eq(True)
    .fillna(False)
)

assert (
    stage1e.loc[
        offer_eligible_mask,
        "catalog_eligible"
    ]
    .eq(True)
    .all()
)

assert (
    stage1e.loc[
        offer_eligible_mask,
        "currently_available"
    ]
    .eq(True)
    .all()
)

assert (
    stage1e.loc[
        offer_eligible_mask,
        "stock_status"
    ]
    == "IN_STOCK"
).all()

assert not (
    (
        stage1e["stock_status"] != "IN_STOCK"
    )
    &
    stage1e["offer_eligible"].eq(True).fillna(False)
).any()

print(
    "PASS: Only catalog-eligible, explicitly in-stock "
    "records are active offer-eligible."
)


PASS: Only catalog-eligible, explicitly in-stock records are active offer-eligible.


## 26. Validate hard-exclusion logic

In [26]:
hard_ineligible_mask = (
    stage1e[
        "eligibility_status"
    ]
    == "INELIGIBLE_HARD_GATE"
)

assert (
    stage1e.loc[
        hard_ineligible_mask,
        "hard_exclusion_reason"
    ]
    != "NONE"
).all()

assert (
    stage1e.loc[
        ~hard_ineligible_mask,
        "hard_exclusion_reason"
    ]
    == "NONE"
).all()

# In revised Stage 1E, hard exclusions should be price-related only.
assert not (
    stage1e["hard_exclusion_reason"]
    .astype("string")
    .str.contains(
        "OUT_OF_STOCK|STOCK_UNKNOWN",
        regex=True,
        na=False
    )
).any()

print(
    "PASS: Hard exclusions are price-quality failures only; "
    "stock is not a catalog hard exclusion."
)


PASS: Hard exclusions are price-quality failures only; stock is not a catalog hard exclusion.


## 27. Validate preliminary catalog eligibility

In [27]:
preliminary_rows = (
    stage1e[
        "eligibility_status"
    ]
    == "ELIGIBLE_PRELIMINARY"
)

assert (
    stage1e.loc[
        preliminary_rows,
        "brand_status"
    ]
    == "VALID"
).all()

assert (
    stage1e.loc[
        preliminary_rows,
        "price_status"
    ]
    == "VALID"
).all()

assert (
    stage1e.loc[
        preliminary_rows,
        "spec_quality_status"
    ]
    == "PARSED"
).all()

assert (
    stage1e.loc[
        preliminary_rows,
        "catalog_eligible"
    ]
    .eq(True)
    .all()
)

print(
    "PASS: Every ELIGIBLE_PRELIMINARY row "
    "passes the Stage 1E catalog-quality gate."
)


PASS: Every ELIGIBLE_PRELIMINARY row passes the Stage 1E catalog-quality gate.


## 28. Validate review-required logic

In [28]:
review_required_mask = (
    stage1e[
        "eligibility_status"
    ]
    == "REVIEW_REQUIRED"
)

# REVIEW_REQUIRED rows must have a usable price.
assert (
    stage1e.loc[
        review_required_mask,
        "price_status"
    ]
    == "VALID"
).all()

# They must have a brand and/or specification issue.
assert (
    (
        stage1e.loc[
            review_required_mask,
            "brand_status"
        ]
        != "VALID"
    )
    |
    (
        stage1e.loc[
            review_required_mask,
            "spec_quality_status"
        ]
        != "PARSED"
    )
).all()

print(
    "PASS: REVIEW_REQUIRED rows have usable price "
    "but still need brand/specification verification."
)


PASS: REVIEW_REQUIRED rows have usable price but still need brand/specification verification.


## 29. Show eligibility and availability by retailer

In [29]:
catalog_status_by_retailer = pd.crosstab(
    stage1e["retailer"],
    stage1e["eligibility_status"],
    margins=True,
)

display(catalog_status_by_retailer)

availability_by_retailer = pd.crosstab(
    stage1e["retailer"],
    stage1e["stock_status"],
    margins=True,
)

display(availability_by_retailer)


eligibility_status,ELIGIBLE_PRELIMINARY,INELIGIBLE_HARD_GATE,REVIEW_REQUIRED,All
retailer,,,,
coldstorage,8995,0,1462,10457
fairprice,39599,2,29801,69402
redmart,85532,0,12247,97779
All,134126,2,43510,177638


stock_status,IN_STOCK,OUT_OF_STOCK,All
retailer,,,
coldstorage,9054,1403,10457
fairprice,64524,4878,69402
redmart,72975,24804,97779
All,146553,31085,177638


## 30. Build retailer quality summary

In [30]:
summary_rows = []

for retailer, df in stage1e.groupby(
    "retailer",
    sort=False
):
    summary_rows.append(
        {
            "retailer": retailer,
            "total_rows": len(df),

            "brand_valid": int(
                (df["brand_status"] == "VALID").sum()
            ),
            "brand_missing": int(
                (df["brand_status"] == "MISSING").sum()
            ),
            "brand_placeholder": int(
                (df["brand_status"] == "PLACEHOLDER").sum()
            ),

            "price_valid": int(
                (df["price_status"] == "VALID").sum()
            ),
            "price_invalid_zero": int(
                (df["price_status"] == "INVALID_ZERO").sum()
            ),
            "price_invalid_negative": int(
                (df["price_status"] == "INVALID_NEGATIVE").sum()
            ),
            "price_invalid_unparsed": int(
                (df["price_status"] == "INVALID_UNPARSED").sum()
            ),
            "price_missing": int(
                (df["price_status"] == "MISSING").sum()
            ),

            "in_stock": int(
                (df["stock_status"] == "IN_STOCK").sum()
            ),
            "out_of_stock": int(
                (df["stock_status"] == "OUT_OF_STOCK").sum()
            ),
            "stock_unknown": int(
                (df["stock_status"] == "UNKNOWN").sum()
            ),

            "spec_parsed": int(
                (df["spec_quality_status"] == "PARSED").sum()
            ),
            "spec_partial": int(
                (df["spec_quality_status"] == "PARTIAL").sum()
            ),
            "spec_ambiguous": int(
                (df["spec_quality_status"] == "AMBIGUOUS").sum()
            ),
            "spec_missing": int(
                (df["spec_quality_status"] == "MISSING").sum()
            ),

            "needs_manual_review": int(
                df["needs_manual_review"]
                .fillna(False)
                .sum()
            ),

            "catalog_eligible": int(
                df["catalog_eligible"]
                .fillna(False)
                .sum()
            ),

            "currently_available": int(
                df["currently_available"]
                .fillna(False)
                .sum()
            ),

            "offer_eligible": int(
                df["offer_eligible"]
                .fillna(False)
                .sum()
            ),

            "eligible_preliminary": int(
                (
                    df["eligibility_status"]
                    == "ELIGIBLE_PRELIMINARY"
                ).sum()
            ),

            "review_required": int(
                (
                    df["eligibility_status"]
                    == "REVIEW_REQUIRED"
                ).sum()
            ),

            "ineligible_hard_gate": int(
                (
                    df["eligibility_status"]
                    == "INELIGIBLE_HARD_GATE"
                ).sum()
            ),
        }
    )

quality_summary = pd.DataFrame(
    summary_rows
)

display(quality_summary)


,retailer,total_rows,brand_valid,brand_missing,brand_placeholder,price_valid,price_invalid_zero,price_invalid_negative,price_invalid_unparsed,price_missing,...,spec_partial,spec_ambiguous,spec_missing,needs_manual_review,catalog_eligible,currently_available,offer_eligible,eligible_preliminary,review_required,ineligible_hard_gate
0,redmart,97779,92660,5119,0,97779,0,0,0,0,...,0,778,6681,12247,85532,72975,64185,85532,12247,0
1,fairprice,69402,68339,1063,0,69400,2,0,0,0,...,27130,1644,234,29803,39599,64524,36971,39599,29801,2
2,coldstorage,10457,10215,18,224,10457,0,0,0,0,...,524,730,0,1462,8995,9054,7841,8995,1462,0


## 31. Build issue-code summary

In [31]:
issue_summary_rows = []

for code_name, mask in ISSUE_MASKS.items():
    issue_summary_rows.append(
        {
            "issue_code": code_name,
            "record_count": int(
                mask.sum()
            ),
            "percentage_of_all_rows": round(
                float(mask.mean() * 100),
                2
            ),
            "hard_exclusion": (
                code_name in HARD_EXCLUSION_CODES
            ),
        }
    )

issue_summary = (
    pd.DataFrame(issue_summary_rows)
    .sort_values(
        "record_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(issue_summary)


,issue_code,record_count,percentage_of_all_rows,hard_exclusion
0,OUT_OF_STOCK,31085,17.50,False
1,SPEC_PARTIAL,27654,15.57,False
2,SPEC_MISSING,6915,3.89,False
3,BRAND_MISSING,6200,3.49,False
4,SPEC_AMBIGUOUS,3152,1.77,False
5,BRAND_PLACEHOLDER,224,0.13,False
6,PRICE_INVALID_ZERO,2,0.00,True
7,PRICE_MISSING,0,0.00,True
8,PRICE_INVALID_UNPARSED,0,0.00,True
9,PRICE_INVALID_NEGATIVE,0,0.00,True


## 32. Create manual-review queue

In [32]:
manual_review_queue = stage1e.loc[
    stage1e[
        "needs_manual_review"
    ]
    .fillna(False),
    [
        "retailer",
        "source_record_id",
        "source_category_raw",
        "product_name_raw",
        "brand_raw",
        "brand_normalized",
        "brand_status",
        "specification_raw",
        "spec_parse_status",
        "spec_parse_pattern",
        "product_price_raw",
        "product_price",
        "price_status",
        "stock_raw",
        "stock_status",
        "quality_issue_codes",
        "hard_exclusion_reason",
        "catalog_eligible",
        "currently_available",
        "offer_eligible",
        "eligibility_status",
        "product_url_raw",
    ]
].copy()

print(
    "Manual-review queue:",
    f"{len(manual_review_queue):,}"
)

display(
    manual_review_queue.head(20)
)


Manual-review queue: 43,512


,retailer,source_record_id,source_category_raw,product_name_raw,brand_raw,brand_normalized,brand_status,specification_raw,spec_parse_status,spec_parse_pattern,...,price_status,stock_raw,stock_status,quality_issue_codes,hard_exclusion_reason,catalog_eligible,currently_available,offer_eligible,eligibility_status,product_url_raw
59,redmart,redmart_00000061,shop-chilled-desserts,The Dark Gallery Pistachio Cranberry Dark Choc...,<NA>,<NA>,MISSING,180 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3342461448...
64,redmart,redmart_00000066,shop-chilled-desserts,The Dark Gallery Artisanal Chocolate Bars - Ha...,<NA>,<NA>,MISSING,45 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3369040051...
71,redmart,redmart_00000073,shop-chilled-desserts,The Dark Gallery Artisanal Chocolate Bar - Ear...,<NA>,<NA>,MISSING,45 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3342439522...
74,redmart,redmart_00000076,shop-chilled-desserts,The Dark Gallery Artisanal Macarons,<NA>,<NA>,MISSING,10 Per Pack,PARSED,COUNT_PER_PACK,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3378835623...
113,redmart,redmart_00000115,shop-chilled-desserts,The Whole Kitchen Gluten Free Energy Balls/ Pr...,Whole Kitchen,whole kitchen,VALID,2 × 12 Per Pack,AMBIGUOUS,UNRECOGNIZED,...,VALID,No,OUT_OF_STOCK,OUT_OF_STOCK;SPEC_AMBIGUOUS,NONE,False,False,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i305124899-...
116,redmart,redmart_00000118,shop-chilled-desserts,The Dark Gallery Artisanal Chocolate Bar - Caf...,<NA>,<NA>,MISSING,45 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3342577281...
117,redmart,redmart_00000119,shop-chilled-desserts,The Dark Gallery Toffee Sea Salt Dark Chocolat...,<NA>,<NA>,MISSING,180 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i3342519369...
157,redmart,redmart_00000159,Eggs,Golden Irish Fresh Free Range Eggs - 10S,<NA>,<NA>,MISSING,10 Per Pack,PARSED,COUNT_PER_PACK,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i1797598693...
189,redmart,redmart_00000191,Eggs,Golden Irish Free Range Organic Eggs 6S,<NA>,<NA>,MISSING,6 Per Pack,PARSED,COUNT_PER_PACK,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i2478563007...
197,redmart,redmart_00000199,Eggs,The Meat Club Fresh Cage Free Eggs (10-pack) |...,<NA>,<NA>,MISSING,680 g,PARSED,SINGLE_METRIC,...,VALID,Yes,IN_STOCK,BRAND_MISSING,NONE,False,True,False,REVIEW_REQUIRED,https://www.lazada.sg/products/pdp-i1380694823...


## 33. Create hard-ineligible queue

In [33]:
hard_ineligible_queue = stage1e.loc[
    stage1e[
        "eligibility_status"
    ]
    == "INELIGIBLE_HARD_GATE"
].copy()

print(
    "Hard-ineligible records:",
    f"{len(hard_ineligible_queue):,}"
)

display(
    hard_ineligible_queue[
        [
            "retailer",
            "source_record_id",
            "source_category_raw",
            "product_name_raw",
            "brand_raw",
            "product_price",
            "price_status",
            "stock_status",
            "hard_exclusion_reason",
            "product_url_raw",
        ]
    ].head(20)
)


Hard-ineligible records: 2


,retailer,source_record_id,source_category_raw,product_name_raw,brand_raw,product_price,price_status,stock_status,hard_exclusion_reason,product_url_raw
110226,fairprice,fairprice_00012449,Chocolate Bars & Blocks,Meiji Chocolate Block - Black,Meiji,0.0,INVALID_ZERO,OUT_OF_STOCK,PRICE_INVALID_ZERO,https://www.fairprice.com.sg/product/meiji-cho...
164532,fairprice,fairprice_00066755,Broths & Stocks,Daisho Crab Miso Hot Pot Soup,Daisho,0.0,INVALID_ZERO,OUT_OF_STOCK,PRICE_INVALID_ZERO,https://www.fairprice.com.sg/product/daisho-cr...


## 34. Create catalog-eligible dataset

In [34]:
catalog_eligible_df = stage1e.loc[
    stage1e["catalog_eligible"]
    .fillna(False)
].copy()

print(
    "Catalog-eligible records:",
    f"{len(catalog_eligible_df):,}"
)

display(
    catalog_eligible_df[
        [
            "retailer",
            "source_record_id",
            "source_category_raw",
            "product_name_raw",
            "brand_normalized",
            "specification_raw",
            "product_price",
            "stock_status",
            "currently_available",
            "offer_eligible",
        ]
    ].head(20)
)


Catalog-eligible records: 134,126


,retailer,source_record_id,source_category_raw,product_name_raw,brand_normalized,specification_raw,product_price,stock_status,currently_available,offer_eligible
0,redmart,redmart_00000002,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun,6 × 100 g,12.0,IN_STOCK,True,True
1,redmart,redmart_00000003,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix,4 × 125 g,10.9,IN_STOCK,True,True
2,redmart,redmart_00000004,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet,400 g,3.0,IN_STOCK,True,True
3,redmart,redmart_00000005,shop-chilled-desserts,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere,4 × 115 g,10.09,IN_STOCK,True,True
4,redmart,redmart_00000006,shop-chilled-desserts,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix,4 × 100 g,7.8,IN_STOCK,True,True
5,redmart,redmart_00000007,shop-chilled-desserts,Fortune Sweetened Chin Chow 1KG,fortune,1 kg,2.85,IN_STOCK,True,True
6,redmart,redmart_00000008,shop-chilled-desserts,RedMart Trio Swiss Roll (Singapore Special),redmart,430 g,10.1,IN_STOCK,True,True
7,redmart,redmart_00000009,shop-chilled-desserts,Arla Protein Chocolate Pudding 200G - Lactose ...,arla,200 g,4.3,IN_STOCK,True,True
8,redmart,redmart_00000010,shop-chilled-desserts,Fortune Soya Beancurd Almond 360G,fortune,360 g,1.86,IN_STOCK,True,True
9,redmart,redmart_00000011,shop-chilled-desserts,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla,200 g,4.3,IN_STOCK,True,True


## 35. Create active offer-eligible dataset

In [35]:
offer_eligible_df = stage1e.loc[
    stage1e["offer_eligible"]
    .fillna(False)
].copy()

print(
    "Active offer-eligible records:",
    f"{len(offer_eligible_df):,}"
)

display(
    offer_eligible_df[
        [
            "retailer",
            "source_record_id",
            "source_category_raw",
            "product_name_raw",
            "brand_normalized",
            "specification_raw",
            "product_price",
            "stock_status",
            "offer_eligible",
        ]
    ].head(20)
)


Active offer-eligible records: 108,997


,retailer,source_record_id,source_category_raw,product_name_raw,brand_normalized,specification_raw,product_price,stock_status,offer_eligible
0,redmart,redmart_00000002,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun,6 × 100 g,12.0,IN_STOCK,True
1,redmart,redmart_00000003,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix,4 × 125 g,10.9,IN_STOCK,True
2,redmart,redmart_00000004,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet,400 g,3.0,IN_STOCK,True
3,redmart,redmart_00000005,shop-chilled-desserts,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere,4 × 115 g,10.09,IN_STOCK,True
4,redmart,redmart_00000006,shop-chilled-desserts,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix,4 × 100 g,7.8,IN_STOCK,True
5,redmart,redmart_00000007,shop-chilled-desserts,Fortune Sweetened Chin Chow 1KG,fortune,1 kg,2.85,IN_STOCK,True
6,redmart,redmart_00000008,shop-chilled-desserts,RedMart Trio Swiss Roll (Singapore Special),redmart,430 g,10.1,IN_STOCK,True
7,redmart,redmart_00000009,shop-chilled-desserts,Arla Protein Chocolate Pudding 200G - Lactose ...,arla,200 g,4.3,IN_STOCK,True
8,redmart,redmart_00000010,shop-chilled-desserts,Fortune Soya Beancurd Almond 360G,fortune,360 g,1.86,IN_STOCK,True
9,redmart,redmart_00000011,shop-chilled-desserts,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla,200 g,4.3,IN_STOCK,True


## 36. Create searchable out-of-stock dataset

In [36]:
catalog_out_of_stock = stage1e.loc[
    stage1e["catalog_eligible"].fillna(False)
    &
    (
        stage1e["stock_status"]
        == "OUT_OF_STOCK"
    )
].copy()

print(
    "Searchable catalog records currently out of stock:",
    f"{len(catalog_out_of_stock):,}"
)

display(
    catalog_out_of_stock[
        [
            "retailer",
            "source_record_id",
            "product_name_raw",
            "brand_normalized",
            "product_price",
            "stock_status",
            "catalog_eligible",
            "currently_available",
            "offer_eligible",
            "product_url_raw",
        ]
    ].head(20)
)


Searchable catalog records currently out of stock: 25,129


,retailer,source_record_id,product_name_raw,brand_normalized,product_price,stock_status,catalog_eligible,currently_available,offer_eligible,product_url_raw
75,redmart,redmart_00000077,Marigold Pudding - Custard Caramel 95G,marigold,1.68,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i301146696-...
76,redmart,redmart_00000078,Monoprix Gourmet Dark Chocolate Lava Cake,monoprix,8.5,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i462678741-...
77,redmart,redmart_00000079,DANONE Danette Chocolate,danone,9.89,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i301150927-...
78,redmart,redmart_00000080,Monoprix Gourmet Dark Chocolate Mousse 2X90G,monoprix,10.7,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i1711625416...
79,redmart,redmart_00000081,Monoprix Tiramisu With Mascarpone 2X80G,monoprix,7.6,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i1711730242...
80,redmart,redmart_00000082,TARAMI Pure Series Peach Jelly - Japan Special,tarami,3.97,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i303414722-...
81,redmart,redmart_00000083,La Laitiere Secret Chocolate Mousse 4 X 59G,la laitiere,6.9,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i303668940-...
82,redmart,redmart_00000084,DANONE Danette Vanilla 4x125g,danone,8.9,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i301172032-...
83,redmart,redmart_00000085,Arla Protein Vanilla Cookie Pudding 200G - Lac...,arla,4.3,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i1372733469...
84,redmart,redmart_00000086,Marigold Jelly - Mango 125G,marigold,1.68,OUT_OF_STOCK,True,False,False,https://www.lazada.sg/products/pdp-i301128791-...


## 37. Verify row counts and provenance remain unchanged

In [37]:
assert len(stage1e) == EXPECTED_TOTAL_ROWS

assert (
    stage1e["source_record_id"]
    .nunique()
    == EXPECTED_TOTAL_ROWS
)

assert (
    stage1e["source_record_id"]
    .duplicated()
    .sum()
    == 0
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1e["retailer"]
        == retailer
    ).sum()

    assert actual_count == expected_count

print(
    "PASS: Stage 1E preserved all rows "
    "and provenance IDs."
)


PASS: Stage 1E preserved all rows and provenance IDs.


## 38. Save retailer-specific Stage 1E checkpoints

In [38]:
for retailer in [
    "redmart",
    "fairprice",
    "coldstorage",
]:
    retailer_df = (
        stage1e.loc[
            stage1e["retailer"]
            == retailer
        ]
        .copy()
    )

    output_file = (
        STAGING_DATA_DIR
        / f"{retailer}_stage1e_quality.csv"
    )

    retailer_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8"
    )

    print(
        f"Saved {retailer}: "
        f"{len(retailer_df):,} rows -> "
        f"{output_file}"
    )


Saved redmart: 97,779 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\redmart_stage1e_quality.csv
Saved fairprice: 69,402 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\fairprice_stage1e_quality.csv
Saved coldstorage: 10,457 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\coldstorage_stage1e_quality.csv


## 39. Save combined Stage 1E dataset

In [39]:
STAGE1E_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1e_quality.csv"
)

stage1e.to_csv(
    STAGE1E_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved combined Stage 1E dataset:")
print(STAGE1E_OUTPUT_FILE)


Saved combined Stage 1E dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1e_quality.csv


## 40. Save Stage 1E reports and queues

In [40]:
QUALITY_SUMMARY_FILE = (
    STAGING_DATA_DIR
    / "stage1e_quality_summary.csv"
)

ISSUE_SUMMARY_FILE = (
    STAGING_DATA_DIR
    / "stage1e_issue_summary.csv"
)

MANUAL_REVIEW_FILE = (
    STAGING_DATA_DIR
    / "stage1e_manual_review.csv"
)

HARD_INELIGIBLE_FILE = (
    STAGING_DATA_DIR
    / "stage1e_hard_ineligible.csv"
)

CATALOG_ELIGIBLE_FILE = (
    STAGING_DATA_DIR
    / "stage1e_catalog_eligible.csv"
)

OFFER_ELIGIBLE_FILE = (
    STAGING_DATA_DIR
    / "stage1e_offer_eligible.csv"
)

OUT_OF_STOCK_CATALOG_FILE = (
    STAGING_DATA_DIR
    / "stage1e_catalog_out_of_stock.csv"
)

ZERO_PRICE_FILE = (
    STAGING_DATA_DIR
    / "stage1e_zero_price_review.csv"
)

quality_summary.to_csv(
    QUALITY_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)

issue_summary.to_csv(
    ISSUE_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)

manual_review_queue.to_csv(
    MANUAL_REVIEW_FILE,
    index=False,
    encoding="utf-8"
)

hard_ineligible_queue.to_csv(
    HARD_INELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)

catalog_eligible_df.to_csv(
    CATALOG_ELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)

offer_eligible_df.to_csv(
    OFFER_ELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)

catalog_out_of_stock.to_csv(
    OUT_OF_STOCK_CATALOG_FILE,
    index=False,
    encoding="utf-8"
)

zero_price_review.to_csv(
    ZERO_PRICE_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved quality summary:")
print(QUALITY_SUMMARY_FILE)

print("\nSaved issue summary:")
print(ISSUE_SUMMARY_FILE)

print("\nSaved manual-review queue:")
print(MANUAL_REVIEW_FILE)

print("\nSaved hard-ineligible queue:")
print(HARD_INELIGIBLE_FILE)

print("\nSaved catalog-eligible dataset:")
print(CATALOG_ELIGIBLE_FILE)

print("\nSaved active offer-eligible dataset:")
print(OFFER_ELIGIBLE_FILE)

print("\nSaved searchable out-of-stock catalog:")
print(OUT_OF_STOCK_CATALOG_FILE)

print("\nSaved zero-price review:")
print(ZERO_PRICE_FILE)


Saved quality summary:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_quality_summary.csv

Saved issue summary:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_issue_summary.csv

Saved manual-review queue:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_manual_review.csv

Saved hard-ineligible queue:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_hard_ineligible.csv

Saved catalog-eligible dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_catalog_eligible.csv

Saved active offer-eligible dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_offer_eligible.csv

Saved searchable out-of-stock catalog:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1e_catalog_out_

## 41. Verify Stage 1E output files

In [41]:
OUTPUT_FILES = [
    (
        STAGING_DATA_DIR
        / "redmart_stage1e_quality.csv"
    ),
    (
        STAGING_DATA_DIR
        / "fairprice_stage1e_quality.csv"
    ),
    (
        STAGING_DATA_DIR
        / "coldstorage_stage1e_quality.csv"
    ),
    STAGE1E_OUTPUT_FILE,
    QUALITY_SUMMARY_FILE,
    ISSUE_SUMMARY_FILE,
    MANUAL_REVIEW_FILE,
    HARD_INELIGIBLE_FILE,
    CATALOG_ELIGIBLE_FILE,
    OFFER_ELIGIBLE_FILE,
    OUT_OF_STOCK_CATALOG_FILE,
    ZERO_PRICE_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:45s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print(
    "\nPASS: All revised Stage 1E output files were created."
)


redmart_stage1e_quality.csv                   -> True
fairprice_stage1e_quality.csv                 -> True
coldstorage_stage1e_quality.csv               -> True
all_retailers_stage1e_quality.csv             -> True
stage1e_quality_summary.csv                   -> True
stage1e_issue_summary.csv                     -> True
stage1e_manual_review.csv                     -> True
stage1e_hard_ineligible.csv                   -> True
stage1e_catalog_eligible.csv                  -> True
stage1e_offer_eligible.csv                    -> True
stage1e_catalog_out_of_stock.csv              -> True
stage1e_zero_price_review.csv                 -> True

PASS: All revised Stage 1E output files were created.


## 42. Final Stage 1E validation

In [42]:
eligible_count = int(
    (
        stage1e[
            "eligibility_status"
        ]
        == "ELIGIBLE_PRELIMINARY"
    ).sum()
)

review_count = int(
    (
        stage1e[
            "eligibility_status"
        ]
        == "REVIEW_REQUIRED"
    ).sum()
)

hard_ineligible_count = int(
    (
        stage1e[
            "eligibility_status"
        ]
        == "INELIGIBLE_HARD_GATE"
    ).sum()
)

catalog_eligible_count = int(
    stage1e["catalog_eligible"]
    .fillna(False)
    .sum()
)

currently_available_count = int(
    stage1e["currently_available"]
    .fillna(False)
    .sum()
)

offer_eligible_count = int(
    stage1e["offer_eligible"]
    .fillna(False)
    .sum()
)

catalog_out_of_stock_count = int(
    (
        stage1e["catalog_eligible"].fillna(False)
        &
        (
            stage1e["stock_status"]
            == "OUT_OF_STOCK"
        )
    ).sum()
)

brand_placeholder_count = int(
    (
        stage1e["brand_status"]
        == "PLACEHOLDER"
    ).sum()
)

brand_missing_count = int(
    (
        stage1e["brand_status"]
        == "MISSING"
    ).sum()
)

zero_price_count = int(
    (
        stage1e["price_status"]
        == "INVALID_ZERO"
    ).sum()
)

print("=" * 82)
print("STAGE 1E FINAL VALIDATION — REVISED")
print("=" * 82)

print(
    f"Total records                    : "
    f"{len(stage1e):,}"
)

print(
    f"Unique source IDs                : "
    f"{stage1e['source_record_id'].nunique():,}"
)

print(
    f"Duplicate source IDs             : "
    f"{stage1e['source_record_id'].duplicated().sum():,}"
)

print(
    f"Number of columns                : "
    f"{len(stage1e.columns)}"
)

print("-" * 82)

print(
    f"Brand placeholders (*)           : "
    f"{brand_placeholder_count:,}"
)

print(
    f"Missing normalized brands        : "
    f"{brand_missing_count:,}"
)

print(
    f"Zero product prices              : "
    f"{zero_price_count:,}"
)

print("-" * 82)

print(
    f"ELIGIBLE_PRELIMINARY             : "
    f"{eligible_count:,}"
)

print(
    f"REVIEW_REQUIRED                  : "
    f"{review_count:,}"
)

print(
    f"INELIGIBLE_HARD_GATE             : "
    f"{hard_ineligible_count:,}"
)

print("-" * 82)

print(
    f"Catalog-eligible records         : "
    f"{catalog_eligible_count:,}"
)

print(
    f"Currently available records      : "
    f"{currently_available_count:,}"
)

print(
    f"Active offer-eligible records    : "
    f"{offer_eligible_count:,}"
)

print(
    f"Catalog-eligible but out of stock: "
    f"{catalog_out_of_stock_count:,}"
)

print("-" * 82)

print(
    f"Manual-review queue              : "
    f"{len(manual_review_queue):,}"
)

print(
    f"Hard-ineligible queue            : "
    f"{len(hard_ineligible_queue):,}"
)

print("=" * 82)

assert len(stage1e) == 177_638
assert stage1e["source_record_id"].nunique() == 177_638
assert stage1e["source_record_id"].duplicated().sum() == 0
assert len(stage1e.columns) == 46

assert (
    eligible_count
    + review_count
    + hard_ineligible_count
    == 177_638
)

assert (
    catalog_eligible_count
    == eligible_count
)

assert (
    offer_eligible_count
    <= catalog_eligible_count
)

assert (
    len(catalog_out_of_stock)
    == catalog_out_of_stock_count
)

assert (
    len(hard_ineligible_queue)
    == hard_ineligible_count
)

assert stage1e[
    STAGE1D_COLUMNS
].equals(stage1d_snapshot)

print()
print(
    "PASS: Revised Stage 1E data-quality, catalog-eligibility, "
    "and offer-availability processing succeeded."
)

print()
print(
    "Important: out-of-stock records can remain catalog-eligible "
    "and searchable, but only explicitly in-stock records can be "
    "offer-eligible for active purchasing/basket comparison."
)


STAGE 1E FINAL VALIDATION — REVISED
Total records                    : 177,638
Unique source IDs                : 177,638
Duplicate source IDs             : 0
Number of columns                : 46
----------------------------------------------------------------------------------
Brand placeholders (*)           : 224
Missing normalized brands        : 6,200
Zero product prices              : 2
----------------------------------------------------------------------------------
ELIGIBLE_PRELIMINARY             : 134,126
REVIEW_REQUIRED                  : 43,510
INELIGIBLE_HARD_GATE             : 2
----------------------------------------------------------------------------------
Catalog-eligible records         : 134,126
Currently available records      : 146,553
Active offer-eligible records    : 108,997
Catalog-eligible but out of stock: 25,129
----------------------------------------------------------------------------------
Manual-review queue              : 43,512
Hard-ineligible que

## Revised Stage 1E result

When the final validation passes:

- all **177,638** source records remain present and traceable;
- raw `*` brand placeholders remain preserved;
- zero / missing / invalid prices are not silently replaced;
- stock is preserved as a separate availability dimension;
- a clean out-of-stock record remains searchable;
- `offer_eligible` is `True` only for clean, explicitly in-stock records;
- brand/specification problems remain reviewable rather than silently guessed.

### Next stage

Proceed to **Stage 2A: source-category mapping into the eight agreed CartOptix categories and cross-category pilot selection**.
